# DuckDB + SLayer, in Python

DuckDB reads a **48 KB CSV straight off a CDN** over `httpfs`; SLayer auto-ingests its schema and answers queries about it. Nothing is copied locally — the DuckDB view points at the URL, and every query reaches back over the wire.

**Prerequisites:** `pip install motley-slayer` (DuckDB ships with it).

## 1. Point DuckDB at the file online

We create a **view** over the remote CSV inside a file-backed DuckDB database. DuckDB auto-installs the `httpfs` extension and streams the file; no rows land on disk. We close the connection right after — DuckDB won't share a read-write file across connections, and SLayer opens its own.

In [1]:
import shutil
import warnings
from pathlib import Path

import duckdb
import pandas as pd

# duckdb-engine warns that it can't reflect indices — irrelevant here.
warnings.filterwarnings("ignore", message=".*reflection on indices.*")

from slayer.async_utils import run_sync
from slayer.client.slayer_client import SlayerClient
from slayer.core.models import DatasourceConfig
from slayer.engine.ingestion import ingest_datasource_idempotent
from slayer.storage.yaml_storage import YAMLStorage

CACHE = Path(".cache/python")
shutil.rmtree(CACHE, ignore_errors=True)
CACHE.mkdir(parents=True)

DB_PATH = (CACHE / "weather.duckdb").resolve()
CSV_URL = "https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv"

con = duckdb.connect(str(DB_PATH))
con.execute(f"CREATE OR REPLACE VIEW weather AS SELECT * FROM '{CSV_URL}'")
n_rows = con.sql("SELECT count(*) FROM weather").fetchone()[0]
assert n_rows == 1461, f"expected 1461 rows, got {n_rows}"
schema = con.sql("DESCRIBE weather").fetchdf()
con.close()
schema

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,precipitation,DOUBLE,YES,None,None,None
2,temp_max,DOUBLE,YES,None,None,None
3,temp_min,DOUBLE,YES,None,None,None
4,wind,DOUBLE,YES,None,None,None
5,weather,VARCHAR,YES,None,None,None


## 2. Auto-ingest the schema, SLayer-side

SLayer introspects the view — off the live URL — and builds a semantic model named `weather`, one column per field with a type. No hand-written model.

In [2]:
storage = YAMLStorage(base_dir=str(CACHE / "models"))
ds = DatasourceConfig(name="weather_db", type="duckdb", database=str(DB_PATH))
run_sync(storage.save_datasource(ds))

run_sync(ingest_datasource_idempotent(datasource=ds, storage=storage))

models = run_sync(storage.list_models(data_source="weather_db"))
assert "weather" in models, f"weather model not ingested; got {models}"

weather = run_sync(storage.get_model(name="weather", data_source="weather_db"))
pd.DataFrame([{"column": c.name, "type": c.type} for c in weather.columns])

,column,type
0,date,DATE
1,precipitation,DOUBLE
2,temp_max,DOUBLE
3,temp_min,DOUBLE
4,wind,DOUBLE
5,weather,TEXT


## 3. A warm-up query

Average high and day count per weather type — a plain grouped aggregation, to confirm the model answers questions. Aggregations are chosen at query time with colon syntax (`temp_max:avg`, `*:count`).

In [3]:
client = SlayerClient(storage=storage)

warmup = client.query_sync(
    {
        "source_model": "weather",
        "dimensions": ["weather"],
        "measures": [
            {"formula": "temp_max:avg", "name": "avg_high"},
            {"formula": "*:count", "name": "days"},
        ],
        "order": [{"column": "days", "direction": "desc"}],
    }
)
pd.DataFrame(warmup.data)

,weather.weather,weather.avg_high,weather.days
0,rain,13.454602,641
1,sun,19.861875,640
2,fog,16.757426,101
3,drizzle,15.926415,53
4,snow,5.573077,26


## 4. The hero query: an aggregate as a dimension, plus a ranking transform

Two query-time features at the month grain, in **one** single-stage query:

- **A dimension computed from an aggregate.** `season` — `CASE WHEN temp_max:avg(partition_by=date) >= 18 THEN 'warm' ELSE 'cool' END` — groups each month *warm* or *cool* by its own average high. That value only exists after aggregating, so SLayer computes it in a synthesized stage and regroups on it (the `partition_by=date` grain follows the monthly time dimension).
- **A ranking transform in a measure.** `rank(precipitation:sum)` orders the months by rainfall, with 1 the wettest.

Ordered wettest-first, the result tells a story: every one of Seattle's rainiest months is *cool*-season.

In [4]:
hero = {
    "source_model": "weather",
    "time_dimensions": [{"dimension": "date", "granularity": "month"}],
    "dimensions": [
        {
            "expression": "CASE WHEN temp_max:avg(partition_by=date) >= 18 THEN 'warm' ELSE 'cool' END",
            "name": "season",
        }
    ],
    "measures": [
        {"formula": "precipitation:sum", "name": "total_rain"},
        {"formula": "rank(precipitation:sum)", "name": "rain_rank"},
    ],
    "order": [{"column": "rain_rank", "direction": "asc"}],
}

result = client.query_sync(hero)
df = pd.DataFrame(result.data)

assert len(df) == 48, f"expected 48 monthly rows, got {len(df)}"
# ordered wettest-first: the rainiest month is cool-season
assert df.iloc[0]["weather.season"] == "cool", "expected the wettest month to be cool-season"
df

,weather.season,weather.date,weather.total_rain,weather.rain_rank
0,cool,2015-12-01,284.5,1
1,cool,2014-03-01,240.0,2
2,cool,2015-11-01,212.6,3
3,cool,2012-11-01,210.5,4
4,cool,2012-03-01,183.0,5
5,cool,2012-12-01,174.0,6
6,cool,2012-01-01,173.3,7
7,cool,2014-10-01,171.5,8
8,cool,2012-10-01,170.3,9
9,warm,2013-09-01,156.8,10


## 5. The SQL SLayer ran

Why use SLayer at all, instead of writing SQL directly? Well, here's the SQL corresponding to the above query. Which one do you think is easier for agents to write, or for humans to audit?

In [5]:
assert result.sql, "expected generated SQL on the response"
print(result.sql)

SELECT
    "weather.season",
    "weather.date",
    "weather.total_rain",
    "weather.rain_rank"
FROM (
WITH _cm_temp_max_avg_partition_by_date AS (
  SELECT
    _stage_inner."weather.date_month" AS "date_month",
    _stage_inner."weather.temp_max_avg_partition_by_date" AS "temp_max_avg_partition_by_date"
  FROM (
    SELECT
      DATE_TRUNC('MONTH', weather.date) AS "weather.date_month",
      CAST(AVG(weather.temp_max) AS DOUBLE) AS "weather.temp_max_avg_partition_by_date"
    FROM weather AS weather
    GROUP BY
      DATE_TRUNC('MONTH', weather.date)
  ) AS _stage_inner
), base AS (
  SELECT
    CASE
      WHEN _cm_temp_max_avg_partition_by_date."temp_max_avg_partition_by_date" >= 18
      THEN 'warm'
      ELSE 'cool'
    END AS "weather.season",
    DATE_TRUNC('MONTH', weather.date) AS "weather.date",
    CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.total_rain"
  FROM weather AS weather
  LEFT JOIN _cm_temp_max_avg_partition_by_date
    ON DATE_TRUNC('MONTH', weather.

---

That's a semantic layer over a file on the internet, in a page of code. See the [command-line version](duckdb_cli_nb.ipynb) for the same demo driven entirely through the `slayer` CLI, the [aggregations](../07_aggregations/aggregations.md) guide for computing dimensions from aggregates, and [formulas](../../concepts/formulas.md) for the full transform vocabulary.